# 06 - Gradient Flow Check

Purpose: confirm gradients are flowing as needed through frontend

Test: with minimal model (frontend -> basic loss) do gradients reach all frontend learnable params?

In [ ]:
# Imports

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

from util import ChirpletFilterbank, BLR, AudioFrontEnd, FS

In [ ]:
# Test

nch = 4
afe = AudioFrontEnd(nch)

# Perturb starting values
with torch.no_grad():
    afe.chirplet_bank.theta_bw.copy_(torch.linspace(-2, 2, nch))
    afe.chirplet_bank.theta_fc.copy_(torch.linspace(-2, 2, nch))
    afe.chirplet_bank.theta_T.copy_(torch.linspace(-2, 2, nch))
    afe.chirplet_bank.theta_degree.copy_(torch.linspace(-2, 2, nch))
    afe.chirplet_bank.theta_sign.copy_(torch.linspace(-2, 2, nch))
    
out = afe(torch.randn(1, 2**13))
print(out.abs().sum(dim=-1))  # per-channel energy, [1, nch]
print(' ')

with torch.no_grad():
    bw, fc, T, degree = afe.chirplet_bank._get_constrained_params()
    k = afe.chirplet_bank._generate_kernels(bw, fc, T, degree)

print("bw:", bw.tolist())
print("fc:", fc.tolist())
print("T:", T.tolist())
print("degree:", degree.tolist())
print()
print("k[0] contains NaN:", torch.isnan(k[0]).any().item())
print("k[0] max abs value:", k[0].abs().max().item())
print("k[0] nonzero count:", (k[0] != 0).sum().item(), "/", k[0].shape[-1])
print(' ')

loss = out.abs().pow(2).sum()
loss.backward()

for name, p in afe.named_parameters():
    grad_status = "None" if p.grad is None else (
        "NaN/Inf" if (torch.isnan(p.grad).any() or torch.isinf(p.grad).any()) else
        "all-zero" if torch.all(p.grad == 0) else "OK"
    )
    print(f"{name:30s} grad shape: {tuple(p.grad.shape)}  status: {grad_status}")

print('')
for name, p in afe.named_parameters():
    print(f"{name}: {p.grad.tolist()}")

## Findings

- Gradients flow correctly through the full pipeline (theta_bw/fc/T/degree/sign) — confirmed nonzero, finite, per-channel-distinct